In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2

name, memory.total [MiB]
Tesla T4, 15360 MiB
               total        used        free      shared  buff/cache   available
Mem:            12Gi       996Mi       8.4Gi       2.2Mi       3.5Gi        11Gi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ai4bharat_asr_marathi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q "nemo_toolkit[asr]>=2.4" "datasets>=3.0" soundfile scipy

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
hf_hub_download("bodhan-ai/indic-transcribe-flex", "nemo/load_nemo.py")
print("Access to the model: OK")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Access to the model: OK


In [ ]:
import torch

ART = "/content/drive/MyDrive/ai4bharat_asr_marathi"   # same folder as cell 2

CFG = {
    # model (Bodhan AI ASR) and prompt
    "model_repo": "bodhan-ai/indic-transcribe-flex",
    "lang": "mr",            # Marathi
    "pnc": "yes",            # same prompt the model card uses at inference
    # data: FLEURS Marathi from Hugging Face, small random subset for free Colab
    "dataset": "google/fleurs",
    "hf_config": "mr_in",
    "text_col": "raw_transcription",   # punctuated text, matches pnc="yes"
    "subset": {"train": 1000, "validation": 150, "test": 300},
    "min_dur": 0.5, "max_dur": 30.0,   # model was trained on clips up to 30 s
    "data_dir": "/content/data",
    # LoRA (same r/alpha/dropout as my Whisper experiments)
    "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
    # training
    "lr": 2e-4, "epochs": 4, "warmup_ratio": 0.1, "min_lr": 1e-6,
    "batch_duration": 40.0,   # seconds of audio per batch (sets GPU memory use)
    "accum": 3,               # gradient accumulation -> ~120 s of audio per update
    "seed": 42,
}

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_gb >= 35:
    CFG.update(batch_duration=160.0, accum=1)
elif gpu_gb >= 20:
    CFG.update(batch_duration=80.0, accum=2)
# bf16 needs an Ampere+ GPU; the T4 uses fp16 mixed precision
PRECISION = "bf16-mixed" if torch.cuda.get_device_capability(0)[0] >= 8 else "16-mixed"
print(f"GPU {torch.cuda.get_device_name(0)} ({gpu_gb:.0f} GB) | precision {PRECISION} | "
      f"batch {CFG['batch_duration']} s x {CFG['accum']} accumulation")

GPU Tesla T4 (15 GB) | precision 16-mixed | batch 40.0 s x 3 accumulation


In [ ]:
import io, json, math, os, re, unicodedata
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio
from scipy.signal import resample_poly

def clean_text(t):
    """Training text: fix encoding noise only (NFC, odd spaces). Punctuation is kept."""
    t = unicodedata.normalize("NFC", t or "")
    t = t.replace("\u00a0", " ").replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", t).strip()

def to_16k_mono(audio_cell):
    """Decode raw bytes with soundfile (avoids torchcodec issues), make mono, resample to 16 kHz."""
    data, sr = sf.read(io.BytesIO(audio_cell["bytes"]), dtype="float32", always_2d=False)
    if data.ndim == 2:
        data = data.mean(axis=1)
    if sr != 16000:
        g = math.gcd(int(sr), 16000)
        data = resample_poly(data, 16000 // g, int(sr) // g).astype(np.float32)
    peak = float(np.abs(data).max()) if data.size else 0.0
    return data / peak if peak > 1.0 else data

def prepare_split(hf_config, lang, split, n):
    """Download one split, take a seeded random subset of n rows, write WAVs + a NeMo manifest."""
    out_dir = f"{CFG['data_dir']}/{hf_config}"
    manifest = f"{out_dir}/{split}_manifest.json"
    if os.path.exists(manifest):
        print(f"{manifest} already exists, skipping")
        return manifest
    ds = load_dataset(CFG["dataset"], hf_config, split=split)
    ds = ds.cast_column("audio", Audio(decode=False))
    if n and n < len(ds):
        # random, not first-n: FLEURS rows are ordered by sentence/speaker
        ds = ds.shuffle(seed=CFG["seed"]).select(range(n))
    os.makedirs(f"{out_dir}/wavs", exist_ok=True)
    rows, dropped = [], {"empty_text": 0, "too_short": 0, "too_long": 0, "decode_error": 0}
    for i, ex in enumerate(ds):
        text = clean_text(ex[CFG["text_col"]])
        if not text:
            dropped["empty_text"] += 1; continue
        try:
            audio = to_16k_mono(ex["audio"])
        except Exception:
            dropped["decode_error"] += 1; continue
        dur = len(audio) / 16000
        if dur < CFG["min_dur"]:
            dropped["too_short"] += 1; continue
        if dur > CFG["max_dur"]:
            dropped["too_long"] += 1; continue
        # FLEURS "id" is a sentence id shared across speakers, so use our own unique id
        utt_id = f"{hf_config}_{split}_{i:05d}"
        path = f"{out_dir}/wavs/{utt_id}.wav"
        sf.write(path, audio, 16000, subtype="PCM_16")
        # source_lang == target_lang tells the Canary model the task is transcription
        rows.append({"audio_filepath": path, "duration": round(dur, 3), "text": text,
                     "source_lang": lang, "target_lang": lang, "pnc": CFG["pnc"], "utt_id": utt_id})
    with open(manifest, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    hours = sum(r["duration"] for r in rows) / 3600
    print(f"{hf_config}/{split}: kept {len(rows)}/{len(ds)} ({hours:.2f} h), dropped {dropped}")
    return manifest

def read_manifest(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

MANIFESTS = {s: prepare_split(CFG["hf_config"], CFG["lang"], s, CFG["subset"][s])
             for s in ["train", "validation", "test"]}
MANIFESTS["hi_test"] = prepare_split("hi_in", "hi", "test", 100)   # for the forgetting check

print("\nExample training lines:")
for r in read_manifest(MANIFESTS["train"])[:2]:
    print(r)

/content/data/mr_in/train_manifest.json already exists, skipping
/content/data/mr_in/validation_manifest.json already exists, skipping
/content/data/mr_in/test_manifest.json already exists, skipping
/content/data/hi_in/test_manifest.json already exists, skipping

Example training lines:
{'audio_filepath': '/content/data/mr_in/wavs/mr_in_train_00000.wav', 'duration': 11.4, 'text': 'किंग सेजोंग हा जोसेओन राजवंशाचा चौथा राजा होता आणि अतिशय सन्माननीय राजांपैकी तो 1 होता.', 'source_lang': 'mr', 'target_lang': 'mr', 'pnc': 'yes', 'utt_id': 'mr_in_train_00000'}
{'audio_filepath': '/content/data/mr_in/wavs/mr_in_train_00001.wav', 'duration': 17.52, 'text': 'अॅड इतर समूहाशी असणाऱ्या संबधावर परिणाम करते कारण इतर मुलांना कळत नाही की ते तसे का वागले आणि त्यांनी ते तशा प्रकारे स्पेल का केले किंवा त्यांची परिपक्वता पटली ही वेगळी आहे.', 'source_lang': 'mr', 'target_lang': 'mr', 'pnc': 'yes', 'utt_id': 'mr_in_train_00001'}


In [ ]:
import sys
from huggingface_hub import snapshot_download

# Only the nemo/ folder is needed (saves ~5 GB of download)
model_dir = snapshot_download(CFG["model_repo"], allow_patterns=["nemo/*", "*.md"])
sys.path.insert(0, f"{model_dir}/nemo")
from load_nemo import load_nemo_model   # ships with the model; registers its custom tokenizer

model = load_nemo_model(model_dir)
# Published in fp16; keep fp32 master weights so small LoRA updates aren't lost to rounding
model = model.float().cuda()
print(type(model).__name__, "| parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6), "M")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[NeMo I 2026-09-22 15:23:40 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-22 15:23:40 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-09-22 15:23:40 mixins:357] Tokenizer SentencePieceTokenizer initialized with 6000 tokens
[NeMo I 2026-09-22 15:23:40 aggregate_tokenizer:73] Aggregate vocab size: 7152


[NeMo W 2026-09-22 15:23:40 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    is_tarred: true
    num_workers: 16
    batch_size: 16
    bucketing_strategy: synced_randomized
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.0
    defer_setup: true
    is_concat: true
    max_duration: 30
    min_duration: 0.025
    pin_memory: true
    shard_manifests: true
    shuffle_n: 2048
    text_field: text
    lang_field: target_lang
    
[NeMo W 2026-09-22 15:23:40 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setu

[NeMo I 2026-09-22 15:24:10 save_restore_connector:287] Model EncDecMultiTaskModel was successfully restored from /root/.cache/huggingface/hub/models--bodhan-ai--indic-transcribe-flex/snapshots/c0db6b2b05409fee7317f193fb207db332a0f14b/nemo/indic_transcribe_flex.nemo.
EncDecMultiTaskModel | parameters: 1221 M


In [ ]:
import contextlib, csv

def normalize_for_scoring(t):
    """For WER only: NFC, drop zero-width chars, remove punctuation (incl. danda), collapse spaces.
    Vowel signs and virama (Unicode 'marks') are KEPT, unlike Whisper's BasicTextNormalizer."""
    t = unicodedata.normalize("NFC", t or "")
    t = "".join(c for c in t if c not in "\u200b\u200c\u200d\ufeff")
    t = "".join(" " if unicodedata.category(c).startswith("P") else c for c in t)
    return re.sub(r"\s+", " ", t.lower()).strip()

def edit_distance(ref, hyp):
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h))
        prev = cur
    return prev[-1]

def score(refs, hyps):
    """Corpus-level WER/CER = total errors / total reference words (or characters)."""
    we = wn = ce = cn = 0
    per_utt = []
    for r, h in zip(refs, hyps):
        rn, hn = normalize_for_scoring(r), normalize_for_scoring(h)
        e = edit_distance(rn.split(), hn.split())
        we += e; wn += len(rn.split())
        ce += edit_distance(list(rn), list(hn)); cn += len(rn)
        per_utt.append(e)
    return {"wer": we / max(wn, 1), "cer": ce / max(cn, 1), "utterances": len(refs)}, per_utt

def transcribe(paths, lang, batch_size=16):
    model.eval()
    ctx = torch.autocast("cuda", dtype=torch.bfloat16) if PRECISION.startswith("bf16") else contextlib.nullcontext()
    with torch.inference_mode(), ctx:
        out = model.transcribe(paths, source_lang=lang, target_lang=lang, pnc=CFG["pnc"],
                               batch_size=batch_size, num_workers=0)
    if isinstance(out, tuple):
        out = out[0]
    return [o.text if hasattr(o, "text") else str(o) for o in out]

def evaluate(manifest, lang, tag):
    """Transcribe a manifest, print WER/CER, save per-utterance results to Drive."""
    rows = read_manifest(manifest)
    hyps = transcribe([r["audio_filepath"] for r in rows], lang)
    summary, per_utt = score([r["text"] for r in rows], hyps)
    summary["tag"] = tag
    os.makedirs(f"{ART}/results", exist_ok=True)
    with open(f"{ART}/results/{tag}.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["utt_id", "word_errors", "ref", "hyp"])
        for r, h, e in zip(rows, hyps, per_utt):
            w.writerow([r["utt_id"], e, r["text"], h])
    json.dump(summary, open(f"{ART}/results/{tag}.json", "w"), indent=2)
    print(f"[{tag}] WER {100 * summary['wer']:.2f}% | CER {100 * summary['cer']:.2f}% | {summary['utterances']} utterances")
    return summary, rows, hyps

# quick self-check of the scorer
print(score(["मी घरी जातो।"], ["मी घरी जातो"])[0])   # punctuation difference only -> WER 0

{'wer': 0.0, 'cer': 0.0, 'utterances': 1}


In [ ]:
baseline_mr, test_rows, baseline_hyps = evaluate(MANIFESTS["test"], "mr", "baseline_mr")
baseline_hi, _, _ = evaluate(MANIFESTS["hi_test"], "hi", "baseline_hi")

[NeMo W 2026-09-22 14:40:43 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-09-22 14:40:43 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-09-22 14:40:43 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 19it [06:49, 21.53s/it]
[NeMo W 2026-09-22 14:47:33 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-09-22 14:47:33 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: trim_silence,

[baseline_mr] WER 18.24% | CER 6.42% | 297 utterances


Transcribing: 7it [01:35, 13.67s/it]


[baseline_hi] WER 11.85% | CER 6.20% | 100 utterances


In [ ]:
import csv
baseline_mr = json.load(open(f"{ART}/results/baseline_mr.json"))
baseline_hi = json.load(open(f"{ART}/results/baseline_hi.json"))
test_rows = read_manifest(MANIFESTS["test"])
with open(f"{ART}/results/baseline_mr.csv", encoding="utf-8") as f:
    hyp_by_id = {r["utt_id"]: r["hyp"] for r in csv.DictReader(f)}
missing = [r["utt_id"] for r in test_rows if r["utt_id"] not in hyp_by_id]
assert not missing, f"Test set differs from the saved baseline ({len(missing)} missing) - re-run Cell 9 instead"
baseline_hyps = [hyp_by_id[r["utt_id"]] for r in test_rows]
print(f"Baseline reloaded: Marathi WER {100*baseline_mr['wer']:.2f}% | Hindi WER {100*baseline_hi['wer']:.2f}%")

Baseline reloaded: Marathi WER 18.24% | Hindi WER 11.85%


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

class LoRALinear(nn.Module):
    """Frozen nn.Linear + trainable low-rank update:  y = W x + (alpha/r) * B(A(dropout(x)))."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.scaling = alpha / r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))   # B = 0 -> starts as the base model
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
    @property
    def in_features(self): return self.base.in_features
    @property
    def out_features(self): return self.base.out_features
    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.lora_dropout(x), self.lora_A), self.lora_B) * self.scaling

# Which layers get LoRA (regexes over NeMo module names)
LORA_TARGETS = {
    "encoder_attn":       r"^encoder\.layers\.\d+\.self_attn\.linear_(q|k|v|out)$",
    "encoder_ffn":        r"^encoder\.layers\.\d+\.feed_forward[12]\.linear[12]$",
    "decoder_self_attn":  r"^transf_decoder\..*\.first_sub_layer\.(query_net|key_net|value_net|out_projection)$",
    "decoder_cross_attn": r"^transf_decoder\..*\.second_sub_layer\.(query_net|key_net|value_net|out_projection)$",
}

def inject_lora(model, r, alpha, dropout):
    for p in model.parameters():          # freeze the whole pretrained model
        p.requires_grad_(False)
    chosen, per_group = [], Counter()
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            for group, rx in LORA_TARGETS.items():
                if re.search(rx, name):
                    chosen.append(name); per_group[group] += 1; break
    if not chosen:
        raise RuntimeError("No layer matched LORA_TARGETS - check the layer list printed above.")
    for name in chosen:
        parent, child = name.rsplit(".", 1)
        setattr(model.get_submodule(parent), child, LoRALinear(getattr(model.get_submodule(parent), child), r, alpha, dropout))
    return per_group

def reset_lora(model):
    """Back to 'no adaptation' (used after the smoke test)."""
    for m in model.modules():
        if isinstance(m, LoRALinear):
            nn.init.kaiming_uniform_(m.lora_A, a=math.sqrt(5)); nn.init.zeros_(m.lora_B)

def lora_state(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if ".lora_" in k}

# Show the model's Linear layer names (layer numbers collapsed to N) to confirm the targets exist
names = Counter(re.sub(r"\.\d+\.", ".N.", n) for n, m in model.named_modules() if isinstance(m, nn.Linear))
for n, c in sorted(names.items()):
    print(f"{c:>4} x {n}")

  32 x encoder.layers.N.feed_forward1.linear1
  32 x encoder.layers.N.feed_forward1.linear2
  32 x encoder.layers.N.feed_forward2.linear1
  32 x encoder.layers.N.feed_forward2.linear2
  32 x encoder.layers.N.self_attn.linear_k
  32 x encoder.layers.N.self_attn.linear_out
  32 x encoder.layers.N.self_attn.linear_pos
  32 x encoder.layers.N.self_attn.linear_q
  32 x encoder.layers.N.self_attn.linear_v
   1 x encoder.pre_encode.out
   1 x log_softmax.mlp.layer0
  24 x transf_decoder._decoder.layers.N.first_sub_layer.key_net
  24 x transf_decoder._decoder.layers.N.first_sub_layer.out_projection
  24 x transf_decoder._decoder.layers.N.first_sub_layer.query_net
  24 x transf_decoder._decoder.layers.N.first_sub_layer.value_net
  24 x transf_decoder._decoder.layers.N.second_sub_layer.key_net
  24 x transf_decoder._decoder.layers.N.second_sub_layer.out_projection
  24 x transf_decoder._decoder.layers.N.second_sub_layer.query_net
  24 x transf_decoder._decoder.layers.N.second_sub_layer.value_net

In [ ]:
per_group = inject_lora(model, CFG["lora_r"], CFG["lora_alpha"], CFG["lora_dropout"])
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print("LoRA layers per group:", dict(per_group))
print(f"Trainable: {n_train/1e6:.1f}M of {n_total/1e6:.0f}M ({100*n_train/n_total:.2f}%)")
missing = [g for g in LORA_TARGETS if per_group[g] == 0]
if missing:
    print("WARNING: these groups matched nothing:", missing, "- tell me and we'll fix the names")

LoRA layers per group: {'encoder_ffn': 128, 'encoder_attn': 128, 'decoder_self_attn': 96, 'decoder_cross_attn': 96}
Trainable: 41.9M of 1263M (3.32%)


In [ ]:
import time
from collections import Counter
import lightning.pytorch as pl
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.strategies import SingleDeviceStrategy
from omegaconf import OmegaConf, open_dict

pl.seed_everything(CFG["seed"])
CFG["epochs"] = 2                  # shorter run so it fits a free Colab session
model.cuda()                       # in case an earlier run left the model on the CPU
reset_lora(model)                  # start LoRA from zero (undo any earlier test steps)

# ---------- 1. Clean the training/validation text for the tokenizer ----------
# Some FLEURS references contain characters this model's tokenizer cannot represent: the
# zero-width joiner (e.g. in "दुसर्‍या") and some digits. They become <unk> tokens, and training
# on <unk> teaches the model to output <unk>. ZWJ/ZWNJ are invisible, so they are removed; any
# sentence that still produces <unk> is left out of TRAINING only. Test sets stay untouched, so
# the comparison with the baseline is fair.
tok = model.tokenizer
def roundtrip(t):
    try:
        ids = tok.text_to_ids(t, CFG["lang"])
    except TypeError:
        ids = tok.text_to_ids(t)
    return tok.ids_to_text(ids)

def clean_manifest(split):
    rows = read_manifest(MANIFESTS[split])
    kept, removed = [], []
    for r in rows:
        t = r["text"].replace("\u200d", "").replace("\u200c", "")
        (removed if "<unk>" in roundtrip(t) else kept).append({**r, "text": t})
    path = MANIFESTS[split].replace("_manifest.json", "_clean_manifest.json")
    with open(path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    chars = Counter(c for r in removed for c in r["text"])
    unk_chars = sorted(c for c in chars if "<unk>" in roundtrip(c))
    print(f"{split}: kept {len(kept)}, left out {len(removed)} with untokenizable characters {unk_chars}")
    return path

MANIFESTS["train_clean"] = clean_manifest("train")
MANIFESTS["val_clean"] = clean_manifest("validation")

# ---------- 2. Dataloader settings ----------
DROP = ["manifest_filepath", "input_cfg", "shar_path", "cuts_path", "tarred_audio_filepaths", "is_tarred",
        "tarred_shard_strategy", "batch_size", "batch_duration", "quadratic_duration", "use_bucketing",
        "num_buckets", "bucket_duration_bins", "bucket_batch_size", "bucket_buffer_size", "shuffle_buffer_size",
        "max_tps", "min_tps", "max_tpb", "min_tpb", "bucketing_2d_strict_mode", "max_open_streams",
        "concurrent_bucketing", "drop_last", "shuffle", "num_workers", "max_duration", "min_duration",
        "use_multimodal_sampling", "batch_tokens", "token_equivalent_duration", "quadratic_factor",
        "measure_total_length"]

def template(key):
    t = model.cfg.get(key)
    if t is None:
        return {}
    try:
        return OmegaConf.to_container(t, resolve=True)
    except Exception:
        return OmegaConf.to_container(t, resolve=False)

if "TRAIN_TEMPLATE" not in globals():   # capture the checkpoint's original settings only once
    TRAIN_TEMPLATE, VAL_TEMPLATE = template("train_ds"), template("validation_ds")

def ds_config(tmpl, manifest, train):
    c = {k: v for k, v in tmpl.items() if k not in DROP}
    c.update(use_lhotse=True, manifest_filepath=manifest, sample_rate=16000, num_workers=2, pin_memory=True,
             min_duration=CFG["min_dur"], max_duration=CFG["max_dur"], shuffle=train,
             text_field="text", lang_field="target_lang")
    if train:   # dynamic batching: fill each batch up to N seconds of audio
        c.update(batch_duration=CFG["batch_duration"], use_bucketing=True, num_buckets=6,
                 bucket_buffer_size=20000, shuffle_buffer_size=10000, seed=CFG["seed"])
    else:
        c.update(batch_size=8, use_bucketing=False)
    return OmegaConf.create(c)

# ---------- 3. Callbacks ----------
class KeepBaseInEval(pl.Callback):
    """Frozen network stays in eval mode (BatchNorm stats in the conv modules must not update).
    Only LoRA dropout trains. The top-level model stays in train mode so SpecAugment stays on."""
    def _apply(self, m):
        for _, child in m.named_children():
            if any(True for _ in child.parameters()):
                child.eval()
        for mod in m.modules():
            if isinstance(mod, LoRALinear):
                mod.lora_dropout.train()
    def on_train_epoch_start(self, trainer, m): self._apply(m)
    def on_train_batch_start(self, trainer, m, batch, idx): self._apply(m)

class LossWatch(pl.Callback):
    """Print the training loss regularly and count NaN/inf values (fp16 overflow check)."""
    def __init__(self):
        self.n, self.bad = 0, 0
    def on_train_batch_end(self, trainer, m, outputs, batch, idx):
        loss = outputs.get("loss") if isinstance(outputs, dict) else outputs
        if loss is None:
            return
        v = float(loss.detach())
        self.n += 1
        if math.isnan(v) or math.isinf(v):
            self.bad += 1
        if self.n <= 3 or self.n % 5 == 0:
            print(f"batch {self.n}: train loss {v:.4f} | NaN/inf so far: {self.bad}/{self.n}")

class SaveBestLoRA(pl.Callback):
    """Save only the LoRA weights to Drive whenever validation WER improves
    (val_wer, because the fp16 validation loss comes out NaN while WER is fine)."""
    def __init__(self, run_dir):
        self.run_dir, self.best = run_dir, float("inf")
        os.makedirs(run_dir, exist_ok=True)
    def on_validation_end(self, trainer, m):
        if trainer.sanity_checking:
            return
        metrics = {k: float(v) for k, v in trainer.callback_metrics.items() if k.startswith("val")}
        with open(f"{self.run_dir}/val_history.jsonl", "a") as f:
            f.write(json.dumps({"step": trainer.global_step, **metrics}) + "\n")
        print(f"\n[step {trainer.global_step}] validation: {metrics}")
        v = metrics.get("val_wer")
        if v is not None and not math.isnan(v) and v < self.best:
            self.best = v
            torch.save({"lora": lora_state(m), "step": trainer.global_step, "val_wer": v, "cfg": CFG},
                       f"{self.run_dir}/best_lora.pt")
            print(f"  -> new best val_wer {v:.4f}, saved {self.run_dir}/best_lora.pt")

class StayOnGPU(SingleDeviceStrategy):
    """Lightning normally copies the whole model (~5 GB) to CPU RAM when training ends.
    Free Colab has only ~12.7 GB RAM, so that copy crashes the session. Skip it."""
    def teardown(self):
        self.precision_plugin.teardown()
        self.accelerator.teardown()
        self.checkpoint_io.teardown()

# ---------- 4. Step budget and trainer ----------
train_hours = sum(r["duration"] for r in read_manifest(MANIFESTS["train_clean"])) / 3600
steps_per_epoch = max(1, int(train_hours * 3600 / (CFG["batch_duration"] * 0.85)) // CFG["accum"])
MAX_STEPS = steps_per_epoch * CFG["epochs"]
VAL_EVERY = max(10, steps_per_epoch // 2)
print(f"{train_hours:.2f} h of training audio | ~{steps_per_epoch} steps/epoch | "
      f"max_steps={MAX_STEPS} | validate every {VAL_EVERY} steps")

def setup_training(run_name, max_steps, val_every, limit_val=10):
    run_dir = f"{ART}/runs/{run_name}"
    trainer = pl.Trainer(
        strategy=StayOnGPU(device=torch.device("cuda:0")),
        accelerator="gpu", devices=1, precision=PRECISION,
        max_steps=max_steps, max_epochs=-1,
        val_check_interval=val_every * CFG["accum"], check_val_every_n_epoch=None,
        limit_val_batches=limit_val, num_sanity_val_steps=2,
        accumulate_grad_batches=CFG["accum"], gradient_clip_val=1.0, log_every_n_steps=5,
        logger=[CSVLogger(run_dir, name="csv"), TensorBoardLogger(run_dir, name="tb")],
        callbacks=[LearningRateMonitor("step"), KeepBaseInEval(), LossWatch(), SaveBestLoRA(run_dir)],
        enable_checkpointing=False, use_distributed_sampler=False, default_root_dir=run_dir,
    )
    model.set_trainer(trainer)      # NeMo order: trainer -> data -> optimizer
    with open_dict(model.cfg):
        model.cfg.train_ds = ds_config(TRAIN_TEMPLATE, MANIFESTS["train_clean"], train=True)
        model.cfg.validation_ds = ds_config(VAL_TEMPLATE, MANIFESTS["val_clean"], train=False)
        model.cfg.optim = OmegaConf.create({
            "name": "adamw", "lr": CFG["lr"], "betas": [0.9, 0.98], "weight_decay": 0.0,
            "sched": {"name": "CosineAnnealing", "warmup_steps": max(1, int(CFG["warmup_ratio"] * max_steps)),
                      "min_lr": CFG["min_lr"], "max_steps": max_steps}})
    model.setup_training_data(model.cfg.train_ds)
    model.setup_validation_data(model.cfg.validation_ds)
    model.setup_optimization(model.cfg.optim)
    return trainer, run_dir

print("Training setup ready")

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


train: kept 903, left out 95 with untokenizable characters ['Õ', 'õ', '०', '१', '२', '३', '४', '५', '६', '७', '८', '९', '—', '‘', '’', '“', '”']
validation: kept 141, left out 9 with untokenizable characters ['०', '१', '२', '३', '४', '६', '७', '८', '९']
3.23 h of training audio | ~114 steps/epoch | max_steps=228 | validate every 57 steps
Training setup ready


In [ ]:
trainer, run_dir = setup_training("smoke", max_steps=10, val_every=5, limit_val=2)
torch.cuda.reset_peak_memory_stats()
trainer.fit(model)
print(f"\nSmoke test OK | peak GPU memory {torch.cuda.max_memory_allocated()/1024**3:.1f} GB "
      f"of {gpu_gb:.0f} GB")
reset_lora(model)   # undo the 10 test steps before the real run

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


[NeMo I 2026-09-22 15:31:53 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-09-22 15:31:53 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: concat_sampling_technique,bucketing_strategy,shard_manifests,concat_sampling_temperature,shuffle_n,is_concat,defer_setup
[NeMo W 2026-09-22 15:31:53 dataloader:894] Note: skip_missing_manifest_entries is set to True. If any of your manifests and tar files are mismatched, the entire tar file will be skipped without warning. It's your responsibility to ensure data integrity with this setting.
[NeMo W 2026-09-22 15:31:53 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-09-22 15:31:53 dataloader:635] Creating a Lhotse DynamicBucketingSampler (max_batch_duration=40.0 max_batch_size=None)
[NeMo I 2026-09-22 15:31:53 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-09-22 15:31:53 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-22 15:31:53 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


...

[NeMo I 2026-09-22 15:33:38 wer:318] 
    
[NeMo I 2026-09-22 15:33:38 wer:319] WER reference: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसा ब्रान्का या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 15:33:38 wer:320] WER predicted: सतराशे पंचावन्नमध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसाब्रांका या नावाने पुन्हा बांधले
[NeMo I 2026-09-22 15:33:50 wer:318] 
    
[NeMo I 2026-09-22 15:33:50 wer:319] WER reference: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही, परंतु या देशामध्ये येणारा मध्य अमेरिकेतील बहुतांश माल हा ड्युटी-फ्री असतो.
[NeMo I 2026-09-22 15:33:50 wer:320] WER predicted: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही परंतु या देशामध्ये येणाऱ्या मध्य अमेरिकेतील बहुतांश माल हा ड्युटी फ्री असतो

[step 5] validation: {'val_loss': nan, 'val_wer': 0.31081080436706543, 'val_bleu': 0.537876307964325}
  -> new best val_wer 0.3108, saved /content/drive/MyDrive/ai4bharat_asr_marathi/runs/smoke/best_l

Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-09-22 15:35:03 wer:318] 
    
[NeMo I 2026-09-22 15:35:03 wer:319] WER reference: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसा ब्रान्का या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 15:35:03 wer:320] WER predicted: सतराशे पंचावन्नमध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसाब्रांका या नावाने पुन्हा बांधले
[NeMo I 2026-09-22 15:35:15 wer:318] 
    
[NeMo I 2026-09-22 15:35:15 wer:319] WER reference: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही, परंतु या देशामध्ये येणारा मध्य अमेरिकेतील बहुतांश माल हा ड्युटी-फ्री असतो.
[NeMo I 2026-09-22 15:35:15 wer:320] WER predicted: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही परंतु या देशामध्ये येणाऱ्या मध्य अमेरिकेतील बहुतांश माल हा ड्युटी फ्री असतो


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=10` reached.



[step 10] validation: {'val_loss': nan, 'val_wer': 0.31081080436706543, 'val_bleu': 0.537876307964325}

Smoke test OK | peak GPU memory 9.7 GB of 15 GB


In [ ]:
trainer, run_dir = setup_training("lora_main", MAX_STEPS, VAL_EVERY)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.fit(model)
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nTraining done in {train_min:.1f} min | peak GPU memory {peak_gb:.1f} GB")

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


[NeMo I 2026-09-22 15:44:54 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-09-22 15:44:54 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: concat_sampling_technique,bucketing_strategy,shard_manifests,concat_sampling_temperature,shuffle_n,is_concat,defer_setup
[NeMo W 2026-09-22 15:44:54 dataloader:894] Note: skip_missing_manifest_entries is set to True. If any of your manifests and tar files are mismatched, the entire tar file will be skipped without warning. It's your responsibility to ensure data integrity with this setting.
[NeMo W 2026-09-22 15:44:54 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-09-22 15:44:54 dataloader:635] Creating a Lhotse DynamicBucketingSampler (max_batch_duration=40.0 max_batch_size=None)
[NeMo I 2026-09-22 15:44:54 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-09-22 15:44:54 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-22 15:44:54 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


...

[NeMo I 2026-09-22 16:07:42 wer:318] 
    
[NeMo I 2026-09-22 16:07:42 wer:319] WER reference: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसा ब्रान्का या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 16:07:42 wer:320] WER predicted: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसाब्रांका या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 16:07:55 wer:318] 
    
[NeMo I 2026-09-22 16:07:55 wer:319] WER reference: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही, परंतु या देशामध्ये येणारा मध्य अमेरिकेतील बहुतांश माल हा ड्युटी-फ्री असतो.
[NeMo I 2026-09-22 16:07:55 wer:320] WER predicted: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही, परंतु या देशामध्ये येणाऱ्या मध्य अमेरिकेतील बहुतांश माल हा ड्युटी फ्री असतो.
[NeMo I 2026-09-22 16:08:32 wer:318] 
    
[NeMo I 2026-09-22 16:08:32 wer:319] WER reference: पिसांची रचना ती संघर्षासाठी वापरण्यात आलेली नसून तापमान नियंत्रित करण्यासाठी किंवा दाखवण्यासाठी वापरली असल्याचे द

Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-09-22 16:20:34 wer:318] 
    
[NeMo I 2026-09-22 16:20:34 wer:319] WER reference: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसा ब्रान्का या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 16:20:34 wer:320] WER predicted: 1755 मध्ये भूकंप झाल्यानंतर फक्त रिकामे सोडण्यासाठी पोर्तुगीजांनी ते नष्ट केले आणि कॅसाब्रांका या नावाने पुन्हा बांधले.
[NeMo I 2026-09-22 16:20:46 wer:318] 
    
[NeMo I 2026-09-22 16:20:46 wer:319] WER reference: आपल्याला हे समजले आहे की नाही की हे मला माहीत नाही, परंतु या देशामध्ये येणारा मध्य अमेरिकेतील बहुतांश माल हा ड्युटी-फ्री असतो.
[NeMo I 2026-09-22 16:20:46 wer:320] WER predicted: आपल्याला हे समजले आहे की नाही की हे मला माहित नाही, परंतु या देशामध्ये येणाऱ्या मध्य अमेरिकेतील बहुतांश माल हा ड्युटी फ्री असतो.
[NeMo I 2026-09-22 16:21:01 wer:318] 
    
[NeMo I 2026-09-22 16:21:01 wer:319] WER reference: पिसांची रचना ती संघर्षासाठी वापरण्यात आलेली नसून तापमान नियंत्रित करण्यासाठी किंवा दाखवण्यासाठी वापरली असल्याचे द

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=228` reached.



Training done in 46.7 min | peak GPU memory 10.2 GB


In [ ]:
best = torch.load(f"{run_dir}/best_lora.pt", map_location="cpu", weights_only=False)
model.load_state_dict(best["lora"], strict=False)
print(f"Loaded best LoRA weights from step {best['step']} (val_wer {best['val_wer']:.4f})")

lora_mr, _, lora_hyps = evaluate(MANIFESTS["test"], "mr", "lora_mr")
lora_hi, _, _ = evaluate(MANIFESTS["hi_test"], "hi", "lora_hi")

print("\n| Model | Marathi WER | Marathi CER | Hindi WER |")
print("|---|---|---|---|")
for name, mr, hi in [("Zero-shot", baseline_mr, baseline_hi), ("LoRA fine-tuned", lora_mr, lora_hi)]:
    print(f"| {name} | {100*mr['wer']:.2f}% | {100*mr['cer']:.2f}% | {100*hi['wer']:.2f}% |")

json.dump({"train_minutes": round(train_min, 1), "peak_gpu_gb": round(peak_gb, 2), "best_step": best["step"],
           "best_val_wer": best["val_wer"], "trainable_params": n_train,
           "baseline_mr": baseline_mr, "lora_mr": lora_mr,
           "baseline_hi": baseline_hi, "lora_hi": lora_hi, "cfg": CFG},
          open(f"{run_dir}/summary.json", "w"), indent=2, ensure_ascii=False)

[NeMo W 2026-09-22 16:35:27 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-09-22 16:35:27 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: enable_chunking,trim_silence
[NeMo W 2026-09-22 16:35:27 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


Loaded best LoRA weights from step 57 (val_wer 0.2083)


Transcribing: 19it [08:23, 26.52s/it]
[NeMo W 2026-09-22 16:43:52 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-09-22 16:43:52 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: enable_chunking,trim_silence
[NeMo W 2026-09-22 16:43:52 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[lora_mr] WER 15.23% | CER 4.51% | 297 utterances


Transcribing: 7it [01:54, 16.38s/it]


[lora_hi] WER 14.08% | CER 8.40% | 100 utterances

| Model | Marathi WER | Marathi CER | Hindi WER |
|---|---|---|---|
| Zero-shot | 18.24% | 6.42% | 11.85% |
| LoRA fine-tuned | 15.23% | 4.51% | 14.08% |


In [ ]:
_, base_err = score([r["text"] for r in test_rows], baseline_hyps)
_, lora_err = score([r["text"] for r in test_rows], lora_hyps)
diff = sorted(range(len(test_rows)), key=lambda i: lora_err[i] - base_err[i])
print(f"Improved: {sum(l < b for b, l in zip(base_err, lora_err))} | "
      f"worse: {sum(l > b for b, l in zip(base_err, lora_err))} | "
      f"same: {sum(l == b for b, l in zip(base_err, lora_err))}\n")
for title, idxs in [("MOST IMPROVED", diff[:3]), ("MOST WORSENED", diff[::-1][:3])]:
    print("=" * 20, title)
    for i in idxs:
        print(f"errors {base_err[i]} -> {lora_err[i]}")
        print("  REF :", test_rows[i]["text"])
        print("  BASE:", baseline_hyps[i])
        print("  LoRA:", lora_hyps[i], "\n")

Improved: 123 | worse: 42 | same: 132

==================== MOST IMPROVED
errors 16 -> 1
  REF : यामुळे 802.11 a, 802.11 b आणि 802.11 g सोबत तुल्यक्षम होण्यास त्याला मागे जाता येईल, तथापि बेस स्टेशनमध्ये दुहेरी रेडियोज आहेत.
  BASE:  यामुळे आठशे दोन पॉईंट अकरा ए आठशे दोन पॉईंट अकरा बी आणि आठशे दोन पॉईंट अकरा जीसोबत तुल्यक्षम होण्यास त्याला मागे जाता येईल तथापि बेस स्टेशनमध्ये दुहेरी रेडिओज आहेत
  LoRA:  यामुळे 802.11 A, 802.11 B आणि 802.11 G सोबत तुल्यक्षम होण्यास त्याला मागे जाता येईल, तथापि बेस स्टेशनमध्ये दुहेरी रेडिओज आहेत. 

errors 10 -> 1
  REF : USA जिम्नॅस्टिक्स आणि USOC यांचे लक्ष्य समान आहे - जिम्नॅस्टिक्स आणि इतर खेळांना खेळाडूंकरिता सुरक्षित, सकारात्मक आणि सशक्त वातावरणात त्यांच्या स्वप्नांना पूर्ण करणे शक्य तितके सुरक्षित बनविणे.
  BASE:  यू एस ए जिम्नॅस्टिक्स आणि यू एस ओ सी यांचे लक्ष समान आहे जिम्नॅस्टिक्स आणि इतर खेळांना खेळाडूं खेळाडूंकरिता सुरक्षित सकारात्मक आणि सशक्त वातावरणात त्यांच्या स्वप्नांना पूर्ण करणे शक्य तितके सुरक्षित बनवणे
  LoRA:  USA जिम्नॅस्टिक्स आणि US